# STARE — Quick Start: Single Scenario Evaluation

This notebook demonstrates how to run a single DRL agent on one of the three
paper scenarios (**turbulence-aware**, **no-turbulence**, **synthetic**).

**Prerequisites:** Install the project first with `pip install -e .` from the repo root.

**Time estimate:** ~10 min per agent on GPU, ~30 min on CPU (1M timesteps).

In [ ]:
import os
import yaml
from stable_baselines3 import PPO

from evaluation.experiment_setup import build_envs, materialize_drl_kwargs, compute_benchmark_series
from evaluation.runner import evaluate_agent

## 1. Choose a Scenario

| Scenario | Experiment Dir | Turbulence Threshold | Test Data |
|----------|---------------|---------------------|-----------|
| `turb` | `experiments/neurips2026_turb/` | 200 | Yahoo Finance |
| `noturb` | `experiments/neurips2026_noturb/` | 1e5 (disabled) | Yahoo Finance |
| `synth` | `experiments/neurips2026_synth/` | 200 | Causal SDE CSV |

In [ ]:
SCENARIO = 'turb'  # Change to 'noturb' or 'synth'

config_path = os.path.join('..', 'experiments', f'neurips2026_{SCENARIO}', 'config.yaml')
with open(config_path) as f:
    config = yaml.safe_load(f)

print(f"Scenario: {SCENARIO}")
print(f"Turbulence threshold: {config.get('turbulence_threshold')}")
print(f"Test period: {config['test_start']} → {config['test_end']}")
print(f"Timesteps: {config['timesteps_per_model']:,}")

## 2. Load Data & Build Environments

In [ ]:
base_dir = os.path.join('..', 'experiments', f'neurips2026_{SCENARIO}')

train_env, test_env = build_envs(config, base_dir=base_dir)

print(f"Obs space: {train_env.observation_space.shape}")
print(f"Action space: {train_env.action_space.shape}")

## 3. Train & Evaluate a DRL Agent

In [ ]:
n_assets = train_env.action_space.shape[0]
ppo_kwargs = materialize_drl_kwargs('ppo', config.get('ppo_params', {}), n_assets)

results = evaluate_agent(
    model_class=PPO,
    model_name='PPO',
    train_env=train_env,
    test_env=test_env,
    total_timesteps=config['timesteps_per_model'],
    model_kwargs=ppo_kwargs,
)

print(f"\n{'='*50}")
print(f"Annualized Return: {results['AR']*100:.2f}%")
print(f"Sharpe Ratio:      {results['SR']:.3f}")
print(f"Sortino Ratio:     {results['Sortino']:.3f}")
print(f"PSR (vs Buy&Hold): {results['PSR']:.4f}")
print(f"DSR:               {results['DSR']:.4f}")
print(f"95% CI:            [{results['CI_Low']:.3f}, {results['CI_High']:.3f}]")

## 4. Plot Results

In [ ]:
import matplotlib.pyplot as plt

account = results['AccountValue']
benchmark = compute_benchmark_series(config, base_dir=base_dir)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(account.index, account.values, label='PPO', linewidth=2)
ax.plot(benchmark.index[:len(account)], benchmark.values[:len(account)],
        label='Buy & Hold', linewidth=2, linestyle='--', alpha=0.7)
ax.set_title(f'PPO vs Buy & Hold — {SCENARIO} scenario')
ax.set_ylabel('Portfolio Value ($)')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()